<img src="https://cros.ec.europa.eu/system/files/inline-images/logo%20official%20estp_0.jpg" height="120px">

# ADVANCED PYTHON FOR OFFICIAL STATISTICS
## ICON-Institute · Cologne · June 22–26, 2026 · Day 3 PM
### [Dr. Christian Kauth](https://www.linkedin.com/in/ckauth/)

---

# 🤖 Notebook 06 — MCP Servers & Interactive Visualisation
## *Advanced Python for Official Statistics*

> *"The best interface is no interface."*

This afternoon we make our statistical toolkit **AI-accessible** via the Model Context Protocol (MCP),
then build interactive Plotly dashboards and geospatial choropleth maps.

| Topic | You will be able to… |
|-------|---------------------|
| **MCP basics** | Understand how LLMs call external tools |
| **MCP server** | Expose AROP and Gini as LLM-callable tools with `fastmcp` |
| **Plotly Express** | Build animated income distributions with one function call |
| **Plotly subplots** | Combine bar charts and line charts in a dashboard |
| **geopandas** | Join NUTS region codes to EU boundary shapefiles |
| **Choropleth maps** | Visualise regional poverty across participant countries |

In [1]:
import os
from pathlib import Path

def find_project_root(start: Path) -> Path:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists():
            return candidate
    raise FileNotFoundError("pyproject.toml not found")

PROJECT_ROOT = find_project_root(Path().resolve())
os.chdir(PROJECT_ROOT)
DATA_DIR    = PROJECT_ROOT / "data"
PARQUET_DIR = DATA_DIR / "parquet"
FIGURES_DIR = PROJECT_ROOT / "figures"
FIGURES_DIR.mkdir(exist_ok=True, parents=True)
print(f"✅ Project root: {PROJECT_ROOT}")


✅ Project root: /home/administrateur/projects/python_advanced


### `pyproject.toml` update — Notebook 06

This notebook adds: **seaborn>=0.13, plotly>=5.18, geopandas>=0.14, mcp>=1.3, httpx>=0.26**

Run the cell below to update `pyproject.toml` and install the new packages.

In [4]:
%%writefile pyproject.toml
# Day 6: adds MCP server, Plotly, geopandas
# New in notebook 06:
#   + seaborn>=0.13
#   + plotly>=5.18
#   + geopandas>=0.14
#   + mcp>=1.3
#   + httpx>=0.26
#   + pywin32>=306  (Windows only — required by mcp)

[build-system]
requires = ["hatchling"]
build-backend = "hatchling.build"

[project]
name = "silc-toolkit"
version = "0.1.0"
description = "Advanced Python for Official Statistics -- EU-SILC Analysis Toolkit"
requires-python = ">=3.11"
authors = [{ name = "Christian Kauth", email = "christian.kauth@unifr.ch" }]
license = { text = "MIT" }

dependencies = [
    "pydantic>=2.5",
    "pandas>=2.1",
    "pyarrow>=14.0",
    "polars>=0.20",
    "requests>=2.31",
    "beautifulsoup4>=4.12",
    "sqlalchemy>=2.0",
    "numpy>=1.26",
    "scipy>=1.11",
    "statsmodels>=0.14",
    "scikit-learn>=1.4",
    "shap<0.52",
    "matplotlib>=3.8",
    "numba>=0.60",
    "seaborn>=0.13",
    "plotly>=5.18",
    "geopandas>=0.14",
    "mcp[cli]>=1.3",
    "httpx>=0.26",
    "pywin32>=306; sys_platform=='win32'",
    "nbformat>=4.2.0"
]

[project.optional-dependencies]
dev = [
    "pytest>=7.4",
    "pytest-cov>=4.1",
    "black>=23.0",
    "ruff>=0.3",
]

[project.urls]
Repository = "https://github.com/icon-institute/silc-toolkit"

[tool.hatch.build.targets.wheel]
packages = ["silc_toolkit"]

[tool.pytest.ini_options]
testpaths = ["tests"]
addopts = "-v --tb=short"

[tool.black]
line-length = 88
target-version = ["py311"]

[tool.ruff]
line-length = 88
target-version = "py311"

[tool.ruff.lint]
select = ["E", "F", "I", "UP"]
ignore = ["E501"]

[tool.ruff.lint.isort]
known-first-party = ["silc_toolkit"]


Overwriting pyproject.toml


In [5]:
# Sync all dependencies declared in pyproject.toml
# Run this after every %%writefile pyproject.toml cell
import subprocess
result = subprocess.run(
    ["uv", "pip", "install", "-e", "."],
    capture_output=True, text=True
)
if result.returncode == 0:
    print("silc-toolkit + notebook 06 deps installed")
else:
    print("uv error:", result.stderr[-500:])

silc-toolkit + notebook 06 deps installed


#### ⚠️ Windows only — `pywin32` post-install step

The `mcp` package uses Windows-native process APIs (`pywintypes`, `win32api`).
`pywin32` is installed automatically by `uv` above, but it requires a **one-time post-install step**
to copy its DLLs into the correct location inside the virtual environment.

Run the cell below. On Windows you may see a dialog saying a DLL is locked by another process —
click **Ignore** and the script will still complete successfully (you'll see the green ✅).
This step is a no-op on macOS/Linux.

**Manual fallback** — if the cell fails, close all Python processes, then run this in a terminal:
```powershell
.venv\Scripts\python.exe .venv\Scripts\pywin32_postinstall.py -install
```


In [ ]:
import sys, subprocess
from pathlib import Path

if sys.platform == "win32":
    # pywin32 needs a post-install step to copy pywintypes*.dll into the venv root.
    # If a dialog appears warning that the DLL is in use, click "Ignore" — the script
    # will still complete and you'll see the green checkmark below.
    postinstall = Path(sys.executable).parent / "pywin32_postinstall.py"
    if postinstall.exists():
        result = subprocess.run(
            [sys.executable, str(postinstall), "-install", "-quiet"],
            capture_output=True, text=True,
        )
        if result.returncode == 0:
            print("✅ pywin32 post-install complete (Windows DLLs registered)")
        else:
            print("⚠️  pywin32 post-install failed:\n", result.stderr[-500:])
    else:
        print("⚠️  pywin32_postinstall.py not found — is pywin32 installed?")
else:
    print("ℹ️  Non-Windows platform — pywin32 step skipped")


In [ ]:
import pandas as pd
import numpy as np
import matplotlib
matplotlib.use("Agg")
import warnings
warnings.filterwarnings("ignore")

from silc_toolkit import load_incomes, arop_rate, gini_coefficient, s80s20_ratio
print(f"   pandas {pd.__version__}  |  numpy {np.__version__}")

---
## 1 · Model Context Protocol (MCP)

### 1.1 What is MCP?

[MCP](https://modelcontextprotocol.io/docs/getting-started/intro) is an open standard (Anthropic, 2024) that lets LLMs call external **tools** —
Python functions, database queries, API calls — in a structured, safe way.

<img src="https://mintcdn.com/mcp/bEUxYpZqie0DsluH/images/mcp-simple-diagram.png?w=2500&fit=max&auto=format&n=bEUxYpZqie0DsluH&q=85&s=dc4ab238184b6c70e06e871681c921c5" width="1000px"/>

```
User → Claude Desktop → MCP Client → MCP Server → silc_toolkit functions
                      ←              ←             ← results (JSON)
```

An MCP **tool** is a Python function annotated with:
- A name and description (what the LLM reads to decide whether to call it)
- Typed parameters with descriptions
- A JSON-serialisable return type

### 1.2 Building a SILC MCP Server

We expose three SILC indicators as MCP tools:
1. `get_arop(country, year)` — at-risk-of-poverty rate
2. `get_gini(country, year)` — Gini coefficient
3. `compare_countries(countries, year)` — side-by-side comparison

In [ ]:
%%writefile silc_toolkit/mcp_server.py
from __future__ import annotations

from pathlib import Path
from typing import Annotated

from mcp.server.fastmcp import FastMCP
from silc_toolkit.indicators import arop_rate, gini_coefficient, s80s20_ratio

# ── Create the MCP application ───────────────────────────────────────────────
mcp = FastMCP(
    name="SILC Statistics Server",
    instructions=(
        "You have access to EU-SILC Public Use File microdata for 20+ EU countries "
        "(2004-2013). Use these tools to answer questions about poverty, inequality, "
        "and living conditions. All data is synthetic — safe for education."
    ),
)

# Lazy-load data directory (set by the caller or use default)
_DATA_DIR: Path | None = None


def _get_data_dir() -> Path:
    global _DATA_DIR
    if _DATA_DIR is None:
        # Walk up from this file to find the project root
        here = Path(__file__).resolve().parent
        for candidate in [here, *here.parents]:
            if (candidate / "pyproject.toml").exists():
                _DATA_DIR = candidate / "data"
                return _DATA_DIR
        raise FileNotFoundError("Cannot locate data/ directory")
    return _DATA_DIR


_COUNTRY_NAMES: dict[str, str] = {
    "austria": "AT", "belgium": "BE", "bulgaria": "BG", "cyprus": "CY",
    "czechia": "CZ", "czech republic": "CZ", "germany": "DE", "denmark": "DK",
    "estonia": "EE", "greece": "EL", "hellas": "EL", "spain": "ES",
    "finland": "FI", "france": "FR", "croatia": "HR", "hungary": "HU",
    "ireland": "IE", "italy": "IT", "lithuania": "LT", "luxembourg": "LU",
    "latvia": "LV", "malta": "MT", "netherlands": "NL", "holland": "NL",
    "romania": "RO", "sweden": "SE", "slovenia": "SI", "slovakia": "SK",
    "united kingdom": "UK", "uk": "UK",
}


def _resolve_country(country: str) -> str:
    """Resolve a full country name or ISO code to the 2-letter ISO code."""
    return _COUNTRY_NAMES.get(country.strip().lower(), country.strip().upper())


def _load(country: str, year: int) -> list[float]:
    from silc_toolkit.loaders import load_incomes
    return load_incomes(_resolve_country(country), year, _get_data_dir())


# ── Tools ────────────────────────────────────────────────────────────────────

@mcp.tool()
def get_arop(
    country: Annotated[str, "ISO 2-letter country code or full name (e.g. 'LU', 'BE', 'Belgium', 'Spain')"],
    year:    Annotated[int, "Survey year 2004-2013"],
    threshold_pct: Annotated[float, "Threshold as fraction of median (default 0.60)"] = 0.60,
) -> dict:
    "Compute the at-risk-of-poverty rate for a country and year."
    iso = _resolve_country(country)
    try:
        incomes = _load(country, year)
        if not incomes:
            return {"error": f"No data for {iso} {year}"}
        rate = arop_rate(incomes, threshold_pct=threshold_pct)
        median = sorted(incomes)[len(incomes) // 2]
        return {
            "country":    iso,
            "year":       year,
            "arop_rate":  round(rate, 4),
            "arop_pct":   round(rate * 100, 1),
            "n_households": len(incomes),
            "median_income": round(median, 0),
            "threshold":  round(median * threshold_pct, 0),
        }
    except FileNotFoundError:
        return {"error": f"No PUF data for country {iso!r}. "
                "Available: AT BE BG CY DE DK EE EL ES FI FR HR HU IE IT LT LU LV MT NL RO SE SI SK"}


@mcp.tool()
def get_gini(
    country: Annotated[str, "ISO 2-letter country code or full name (e.g. 'LU', 'Luxembourg')"],
    year:    Annotated[int, "Survey year 2004-2013"],
) -> dict:
    "Compute the Gini coefficient of income inequality for a country and year."
    iso = _resolve_country(country)
    try:
        incomes = _load(country, year)
        if not incomes:
            return {"error": f"No data for {iso} {year}"}
        g = gini_coefficient(incomes)
        s = s80s20_ratio(incomes)
        return {
            "country": iso,
            "year":    year,
            "gini":    round(g, 4),
            "s80s20":  round(s, 2),
            "interpretation": (
                "low inequality (Gini < 0.28)" if g < 0.28 else
                "moderate inequality (Gini 0.28-0.35)" if g < 0.35 else
                "high inequality (Gini > 0.35)"
            ),
        }
    except FileNotFoundError:
        return {"error": f"No PUF data for {iso!r}"}


@mcp.tool()
def compare_countries(
    countries: Annotated[list[str], "List of country codes or full names to compare (e.g. ['BE', 'Belgium', 'ES'])"],
    year:      Annotated[int, "Survey year 2004-2013"],
) -> dict:
    "Compare AROP rate and Gini across multiple EU countries for a given year."
    results = []
    for country in countries:
        iso = _resolve_country(country)
        try:
            incomes = _load(country, year)
            if incomes:
                results.append({
                    "country":  iso,
                    "arop_pct": round(arop_rate(incomes) * 100, 1),
                    "gini":     round(gini_coefficient(incomes), 4),
                    "s80s20":   round(s80s20_ratio(incomes), 2),
                    "n_hh":     len(incomes),
                })
        except FileNotFoundError:
            results.append({"country": iso, "error": "No data"})

    results.sort(key=lambda x: x.get("arop_pct", 999))
    return {"year": year, "countries": results}


@mcp.tool()
def get_trend(
    country: Annotated[str, "ISO 2-letter country code or full name (e.g. 'LU', 'Luxembourg')"],
    start_year: Annotated[int, "First year (2004-2012)"] = 2008,
    end_year:   Annotated[int, "Last year (2005-2013)"] = 2013,
) -> dict:
    "Get AROP trend for a country across multiple years."
    iso = _resolve_country(country)
    trend = []
    for year in range(start_year, end_year + 1):
        try:
            incomes = _load(country, year)
            if incomes:
                trend.append({"year": year, "arop_pct": round(arop_rate(incomes) * 100, 1)})
        except FileNotFoundError:
            pass
    if not trend:
        return {"error": f"No data for {iso} in {start_year}-{end_year}"}

    arop_values = [t["arop_pct"] for t in trend]
    change = arop_values[-1] - arop_values[0]
    return {
        "country": iso,
        "trend":   trend,
        "change_pp": round(change, 1),
        "direction": "improving" if change < 0 else "worsening" if change > 0 else "stable",
    }


if __name__ == "__main__":
    # Run as stdio MCP server: python -m silc_toolkit.mcp_server
    mcp.run(transport="stdio")


In [ ]:
# Test the MCP tools locally (without Claude Desktop)
import importlib, sys

# Reload silc_toolkit to pick up latest changes
for mod in list(sys.modules):
    if mod.startswith("silc_toolkit"):
        del sys.modules[mod]

from silc_toolkit.mcp_server import get_arop, get_gini, compare_countries, get_trend

# Test: AROP for participant countries
print("=== get_arop ===")
for country in ["LU", "IE", "HU", "SE"]:
    result = get_arop(country, 2012)
    if "error" not in result:
        print(f"  {result['country']} 2012: AROP = {result['arop_pct']}%  "
              f"(n={result['n_households']:,}  median=EU{result['median_income']:,.0f})")
    else:
        print(f"  {country}: {result['error']}")

# Test: compare countries
print()
print("=== compare_countries ===")
comp = compare_countries(["BE", "ES", "FR", "HU", "IE", "SE"], 2012)
for row in comp["countries"]:
    if "error" not in row:
        print(f"  {row['country']}: AROP={row['arop_pct']}%  Gini={row['gini']}")

# Test: trend
print()
print("=== get_trend for Luxembourg ===")
trend = get_trend("LU", 2008, 2013)
for t in trend["trend"]:
    print(f"  {t['year']}: {t['arop_pct']}%")
print(f"  Change 2008-2013: {trend['change_pp']:+.1f} pp ({trend['direction']})")

---
### 1.3 Connecting to VS Code Copilot (Agent Mode)

VS Code Copilot supports MCP servers via a `.vscode/mcp.json` file in the workspace root.
Once present, VS Code detects it automatically — you may see a **"Start server"** notification,
or the server starts on first use in Agent mode.

Run the cell below to write the config file. Then:
1. Open **Copilot Chat** (`Ctrl+Alt+I`)
2. Switch to **Agent** mode (dropdown next to the chat input)
3. Ask e.g. *"What is the AROP rate for Greece in 2012?"*

> **macOS / Linux:** replace `Scripts/mcp.exe` with `bin/mcp` in the command path.


In [ ]:
import json

vscode_dir = PROJECT_ROOT / ".vscode"
vscode_dir.mkdir(exist_ok=True)

mcp_config = {
    "servers": {
        "silc-statistics": {
            "type": "stdio",
            "command": "${workspaceFolder}/.venv/Scripts/mcp.exe",
            "args": [
                "run",
                "${workspaceFolder}/silc_toolkit/mcp_server.py",
            ],
            "env": {
                "PYTHONPATH": "${workspaceFolder}",
            },
        }
    }
}

config_path = vscode_dir / "mcp.json"
config_path.write_text(json.dumps(mcp_config, indent=2))
print(f"✅ Written: {config_path}")
print("   Open Copilot Chat → Agent mode to use the SILC MCP server")


---
### 1.4 Connecting to Claude Desktop

Add the following entry to your Claude Desktop config file:
- **Windows:** `%APPDATA%\Claude\claude_desktop_config.json`
- **macOS:** `~/Library/Application Support/Claude/claude_desktop_config.json`

```json
{
  "mcpServers": {
    "silc-statistics": {
      "command": "C:\\path\\to\\python_advanced\\.venv\\Scripts\\mcp.exe",
      "args": [
        "run",
        "C:\\path\\to\\python_advanced\\silc_toolkit\\mcp_server.py"
      ],
      "env": {
        "PYTHONPATH": "C:\\path\\to\\python_advanced"
      }
    }
  }
}
```

Run the cell below to print the config with your actual project path filled in — ready to copy-paste.


In [ ]:
import json, sys
from pathlib import Path

venv_scripts = PROJECT_ROOT / ".venv" / ("Scripts" if sys.platform == "win32" else "bin")
mcp_exe      = venv_scripts / ("mcp.exe" if sys.platform == "win32" else "mcp")
server_py    = PROJECT_ROOT / "silc_toolkit" / "mcp_server.py"

claude_config = {
    "mcpServers": {
        "silc-statistics": {
            "command": str(mcp_exe),
            "args": ["run", str(server_py)],
            "env": {"PYTHONPATH": str(PROJECT_ROOT)},
        }
    }
}

print("Add this to your Claude Desktop config file:")
print(f"  Windows : %APPDATA%\\Claude\\claude_desktop_config.json")
print(f"  macOS   : ~/Library/Application Support/Claude/claude_desktop_config.json")
print()
print(json.dumps(claude_config, indent=2))
print()
print("After saving, restart Claude Desktop and try:")
print('  "What is the AROP rate for France in 2012?"')
print('  "Compare poverty across Belgium, Spain, and Hungary in 2012"')
print('  "Is Luxembourg\'s poverty rate improving or worsening over 2008–2013?"')


<img src="https://kauthentechstorage.blob.core.windows.net/notebookimages/20260623_ICON_Python_MCP_Claude_01.png" width="1000px">

### 1.5 MCP Apps

You can even create Interactive UI applications that render inside MCP hosts like Claude Desktop! Knock yourselves out 🥳 https://modelcontextprotocol.io/extensions/apps/overview

In [ ]:
from IPython.display import HTML, display

video_id = "B7ShoCp2kt8"
display(HTML(f'''
<a href="https://youtube.com/shorts/{video_id}" target="_blank">
  <img src="https://img.youtube.com/vi/{video_id}/0.jpg"
       style="height:560px; cursor:pointer;"
       title="Click to open on YouTube"/>
</a>
'''))

---
## 2 · Interactive Visualisation with Plotly

### 2.1 Why Plotly?

- **Interactive:** hover, zoom, filter — without writing JavaScript
- **Publication-ready:** export to SVG, PNG, HTML
- **Dashboard-ready:** embed in Dash, Streamlit, or standalone HTML
- **Works in Jupyter:** `fig.show()` displays inline

We build three visualisations around the EU-SILC AROP data:
1. Animated income distribution by country and year
2. Country comparison bar chart
3. AROP trend lines

In [ ]:
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# Build the multi-country, multi-year summary dataframe
PARTICIPANT_COUNTRIES = ["BE", "ES", "FR", "EL", "HU", "IE", "LU", "SE"]
YEARS = list(range(2008, 2014))

records = []
for country in PARTICIPANT_COUNTRIES:
    for year in YEARS:
        try:
            inc = load_incomes(country, year, DATA_DIR)
            if inc:
                inc_arr = sorted(inc)
                n       = len(inc_arr)
                records.append({
                    "country": country,
                    "year":    year,
                    "n_hh":   n,
                    "median":  round(inc_arr[n//2], 0),
                    "arop":    round(arop_rate(inc) * 100, 1),
                    "gini":    round(gini_coefficient(inc), 4),
                    "s80s20":  round(s80s20_ratio(inc), 2),
                    "p10":     round(inc_arr[int(n*0.10)], 0),
                    "p90":     round(inc_arr[int(n*0.90)], 0),
                })
        except FileNotFoundError:
            pass

df_viz = pd.DataFrame(records)
print(f"Visualisation dataset: {len(df_viz)} country-year observations")
print(df_viz.pivot_table(index="country", columns="year", values="arop").round(1).to_string())

### 2.2 Barplot

AROP by country (2012), coloured by Gini

In [ ]:
df_2012 = df_viz[df_viz["year"] == 2012].sort_values("arop", ascending=True)

fig_bar = px.bar(
    df_2012,
    x="country",
    y="arop",
    color="gini",
    color_continuous_scale="RdYlGn_r",
    text="arop",
    title="At-Risk-of-Poverty Rate by Country (2012)",
    labels={"arop": "AROP rate (%)", "country": "Country", "gini": "Gini"},
    template="plotly_white",
    hover_data={"median": ":,.0f", "s80s20": ":.2f"},
)
fig_bar.update_traces(texttemplate="%{text:.1f}%", textposition="outside")
fig_bar.update_layout(
    coloraxis_colorbar_title="Gini",
    yaxis_title="AROP rate (%)",
    height=450,
)
fig_bar.write_html(str(FIGURES_DIR / "arop_2012_bar.html"))
fig_bar.show()
print("Saved: arop_2012_bar.html")

### 2.3 Lineplot

AROP trend lines per country (2008-2013)

In [ ]:
country_name = {
    "BE": "Belgium", "ES": "Spain", "FR": "France", "EL": "Greece",
    "HU": "Hungary", "IE": "Ireland", "LU": "Luxembourg", "SE": "Sweden"
}
df_viz_named = df_viz.copy()
df_viz_named["Country"] = df_viz_named["country"].map(country_name)

fig_trend = px.line(
    df_viz_named,
    x="year",
    y="arop",
    color="Country",
    markers=True,
    title="AROP Rate Trend 2008-2013 — Participant Countries",
    labels={"arop": "AROP rate (%)", "year": "Year"},
    template="plotly_white",
    hover_data={"gini": ":.4f", "median": ":,.0f"},
)
fig_trend.update_layout(height=450, hovermode="x unified")
fig_trend.write_html(str(FIGURES_DIR / "arop_trend.html"))
fig_trend.show()
print("Saved: arop_trend.html")

### 2.4. Simple Dashboard

Combined dashboard — AROP + Gini side by side.

For more advanced interactive dashboards, have a look at [Dash](https://plotly.com/dash/).

In [ ]:
fig_dash = make_subplots(
    rows=1, cols=2,
    subplot_titles=["AROP rate (%)", "Gini coefficient"],
    shared_yaxes=False,
)

colors = px.colors.qualitative.Set2
for i, country in enumerate(sorted(df_viz["country"].unique())):
    df_c = df_viz[df_viz["country"] == country].sort_values("year")
    color = colors[i % len(colors)]
    fig_dash.add_trace(
        go.Scatter(x=df_c["year"], y=df_c["arop"],
                   mode="lines+markers", name=country,
                   line=dict(color=color),
                   legendgroup=country, showlegend=True),
        row=1, col=1,
    )
    fig_dash.add_trace(
        go.Scatter(x=df_c["year"], y=df_c["gini"],
                   mode="lines+markers", name=country,
                   line=dict(color=color),
                   legendgroup=country, showlegend=False),
        row=1, col=2,
    )

fig_dash.update_layout(
    title="EU-SILC Poverty & Inequality Dashboard — Participant Countries",
    template="plotly_white",
    height=450,
    hovermode="x unified",
)
fig_dash.write_html(str(PROJECT_ROOT / "silc_dashboard.html"))
fig_dash.show()
print("Saved: silc_dashboard.html")

---
## 3 · Geospatial Analysis with geopandas

### 3.1 Joining SILC Data to EU Boundaries

`geopandas` extends pandas with a geometry column — each row has a geographic shape
(point, line, or polygon). We join our SILC country-level data to the EU NUTS0
(country) boundary shapefile for choropleth maps.

Data source: Eurostat GISCO (Geographic Information System of the Commission)
`https://gisco-services.ec.europa.eu/distribution/v2/nuts/geojson/NUTS_RG_20M_2021_4326.geojson`

In [ ]:
try:
    import geopandas as gpd
    import json
    import requests

    # Download NUTS0 boundaries (country level) from Eurostat GISCO
    GISCO_URL = (
        "https://gisco-services.ec.europa.eu/distribution/v2/nuts/geojson/"
        "NUTS_RG_20M_2021_4326.geojson"
    )
    geo_cache = PROJECT_ROOT / ".scrape_cache" / "nuts0_2021.geojson"
    geo_cache.parent.mkdir(exist_ok=True)

    if not geo_cache.exists():
        print("Downloading NUTS boundaries from Eurostat GISCO...")
        resp = requests.get(GISCO_URL, timeout=30)
        resp.raise_for_status()
        geo_cache.write_bytes(resp.content)
        print(f"  Saved {len(resp.content)//1024} KB")
    else:
        print("Using cached NUTS boundaries")

    gdf = gpd.read_file(str(geo_cache))
    # Keep only NUTS level 0 (countries)
    gdf0 = gdf[gdf["LEVL_CODE"] == 0].copy()
    # NUTS0 code = country code (ISO 3166-1 alpha-2 convention, same as SILC)
    gdf0 = gdf0.rename(columns={"NUTS_ID": "country"})

    print(f"  Loaded {len(gdf0)} country geometries")
    print(f"  CRS: {gdf0.crs}")

except Exception as e:
    print(f"geopandas/network error: {e}")
    gdf0 = None

gdf0.sample(5)

In [ ]:
%matplotlib inline

import matplotlib.pyplot as plt

it = gdf0.loc[gdf0["country"] == "IT"]

fig, ax = plt.subplots(figsize=(8, 8))
it.plot(ax=ax, edgecolor="black", facecolor="lightblue")
ax.set_axis_off()
plt.show()

In [ ]:
try:
    import matplotlib.pyplot as plt
    import json

    # Guard: reset so downstream cells can check availability
    geojson = None
    all_eu_countries = []
    gdf_eu = None

    if gdf0 is not None:
        # Merge SILC 2012 data onto geometries
        df_map = df_viz[df_viz["year"] == 2012][["country", "arop", "gini"]].copy()
        gdf_merged = gdf0.merge(df_map, on="country", how="left")

        # Clip to mainland Europe (drop overseas territories)
        eu_extent = {"minx": -25, "maxx": 40, "miny": 34, "maxy": 72}
        gdf_eu = gdf_merged.cx[
            eu_extent["minx"]:eu_extent["maxx"],
            eu_extent["miny"]:eu_extent["maxy"]
        ]
        print(gdf_eu.keys())

        # Full GeoJSON and list of all clipped country codes
        geojson = json.loads(gdf_eu.to_json())
        all_eu_countries = gdf_eu["country"].dropna().tolist()

        # ── Shared Plotly helpers ─────────────────────────────────────────────
        _GEO_ARGS = dict(
            projection_type="mercator",
            fitbounds="locations", visible=False,
            lataxis_range=[34, 72], lonaxis_range=[-25, 40],
        )
        _MARKER = dict(line_width=0.5, line_color="white")

        def _base_trace(countries):
            """Grey polygon for every EU country (shown even without SILC data)."""
            return go.Choropleth(
                geojson=geojson,
                locations=countries,
                z=[0] * len(countries),
                featureidkey="properties.country",
                colorscale=[[0, "#d3d3d3"], [1, "#d3d3d3"]],
                showscale=False,
                marker=_MARKER,
                hovertemplate="%{location}<br>No data<extra></extra>",
                name="No data",
            )

        print(f"✅ {len(all_eu_countries)} EU country geometries prepared")
        print(f"   SILC 2012 countries in dataset: {sorted(df_map['country'].tolist())}")
    else:
        print("gdf0 not available — skipping geospatial cells")

except Exception as e:
    import traceback
    traceback.print_exc()


### 3.2 Static Map — matplotlib

`geopandas.GeoDataFrame.plot()` renders directly onto a `matplotlib` axes.
Countries without SILC data appear in light grey via `missing_kwds`.
The figure is saved as a high-resolution PNG for reports and slides.


In [ ]:
if gdf_eu is not None:
    fig_mpl, axes = plt.subplots(1, 2, figsize=(14, 6))
    for ax, col, title, cmap in zip(
        axes,
        ["arop", "gini"],
        ["At-risk-of-poverty rate (%) — 2012", "Gini coefficient — 2012"],
        ["YlOrRd", "Blues"],
    ):
        gdf_eu.plot(
            column=col, cmap=cmap, legend=True,
            missing_kwds={"color": "lightgrey", "label": "No data"},
            ax=ax, edgecolor="white", linewidth=0.5,
        )
        ax.set_xlim(-25, 40)
        ax.set_ylim(34, 72)
        ax.set_title(title, fontsize=11, fontweight="bold")
        ax.axis("off")
    plt.suptitle("EU-SILC 2012 — Poverty & Inequality Maps", fontsize=13)
    plt.tight_layout()
    plt.savefig(FIGURES_DIR / "silc_choropleth.png", dpi=120, bbox_inches="tight")
    print("Saved: silc_choropleth.png")
else:
    print("Skipped — gdf_eu not available")



### 3.3 Interactive Choropleths — AROP & Gini (2012)

`go.Choropleth` uses a **two-trace** pattern to show all EU country outlines:

1. **Grey base trace** — all EU polygons with a flat `#d3d3d3` colorscale (`showscale=False`)
2. **Data trace** — SILC countries coloured by the indicator value

`featureidkey="properties.country"` links each data row to the correct GeoJSON feature.
Both maps are saved as standalone HTML for sharing.


In [ ]:
geojson

In [ ]:
df_map

In [ ]:
if geojson is not None:
    # ── AROP 2012 ─────────────────────────────────────────────────────────────
    fig_arop = go.Figure([
        _base_trace(all_eu_countries),
        go.Choropleth(
            geojson=geojson,
            locations=df_map["country"].tolist(),
            z=df_map["arop"].tolist(),
            featureidkey="properties.country",
            colorscale="YlOrRd",
            zmin=df_map["arop"].min(), zmax=df_map["arop"].max(),
            marker=_MARKER,
            colorbar_title="AROP (%)",
            hovertemplate="%{location}<br>AROP: %{z:.1f}%<extra></extra>",
        ),
    ])
    fig_arop.update_geos(**_GEO_ARGS)
    fig_arop.update_layout(
        title="At-Risk-of-Poverty Rate (%) — EU-SILC 2012",
        template="plotly_white", height=700,
        margin=dict(l=0, r=0, t=40, b=0),
    )
    fig_arop.write_html(str(PROJECT_ROOT / "silc_choropleth_arop.html"))
    fig_arop.show()
    print("Saved: silc_choropleth_arop.html")

    # ── Gini 2012 ─────────────────────────────────────────────────────────────
    fig_gini = go.Figure([
        _base_trace(all_eu_countries),
        go.Choropleth(
            geojson=geojson,
            locations=df_map["country"].tolist(),
            z=df_map["gini"].tolist(),
            featureidkey="properties.country",
            colorscale="Blues",
            zmin=df_map["gini"].min(), zmax=df_map["gini"].max(),
            marker=_MARKER,
            colorbar_title="Gini",
            hovertemplate="%{location}<br>Gini: %{z:.4f}<extra></extra>",
        ),
    ])
    fig_gini.update_geos(**_GEO_ARGS)
    fig_gini.update_layout(
        title="Gini Coefficient — EU-SILC 2012",
        template="plotly_white", height=500,
        margin=dict(l=0, r=0, t=40, b=0),
    )
    fig_gini.write_html(str(PROJECT_ROOT / "silc_choropleth_gini.html"))
    fig_gini.show()
    print("Saved: silc_choropleth_gini.html")
else:
    print("Skipped — geojson not available")


### 3.4 Animated Choropleth — AROP 2008–2013

`go.Frame` objects drive the animation — each frame holds two traces:
the grey base layer plus the SILC data coloured for that year.

The `sliders` and `updatemenus` layout keys add a year scrubber and play/pause buttons.
The fixed `zmin`/`zmax` keeps the colour scale stable across all frames.


In [ ]:
if geojson is not None:
    sorted_years = sorted(df_viz["year"].unique())

    def _year_traces(year):
        df_y = df_viz[df_viz["year"] == year]
        return [
            _base_trace(all_eu_countries),
            go.Choropleth(
                geojson=geojson,
                locations=df_y["country"].tolist(),
                z=df_y["arop"].tolist(),
                featureidkey="properties.country",
                colorscale="YlOrRd",
                zmin=df_viz["arop"].min(), zmax=df_viz["arop"].max(),
                marker=_MARKER,
                colorbar_title="AROP (%)",
                hovertemplate="%{location}<br>AROP: %{z:.1f}%<extra></extra>",
                showscale=True,
            ),
        ]

    frames = [go.Frame(data=_year_traces(y), name=str(y)) for y in sorted_years]

    fig_anim = go.Figure(data=_year_traces(sorted_years[0]), frames=frames)
    fig_anim.update_geos(**_GEO_ARGS)
    fig_anim.update_layout(
        title="AROP Rate Evolution 2008–2013 — EU-SILC",
        template="plotly_white", height=540,
        margin=dict(l=0, r=0, t=40, b=60),
        updatemenus=[dict(
            type="buttons", showactive=False,
            y=-0.05, x=0.5, xanchor="center",
            buttons=[
                dict(label="▶ Play", method="animate",
                     args=[None, {"frame": {"duration": 900, "redraw": True},
                                  "fromcurrent": True}]),
                dict(label="⏸ Pause", method="animate",
                     args=[[None], {"frame": {"duration": 0, "redraw": False},
                                    "mode": "immediate"}]),
            ],
        )],
        sliders=[dict(
            active=0,
            steps=[dict(
                args=[[f.name], {"frame": {"duration": 300, "redraw": True},
                                 "mode": "immediate"}],
                label=f.name, method="animate",
            ) for f in frames],
            x=0.1, len=0.8, y=-0.02,
        )],
    )
    fig_anim.write_html(str(PROJECT_ROOT / "silc_choropleth_animated.html"))
    fig_anim.show()
    print("Saved: silc_choropleth_animated.html")
else:
    print("Skipped — geojson not available")


---
### 🏋️ Exercise 1 — Your Country's Trend in Plotly

Choose your participant country (or any country with PUF data). Create:
1. A **bar chart** showing AROP by year (2008-2013)
2. Add a horizontal line at the **EU AROP target** (set this to 15% as a fictional target)
3. Colour bars red if AROP > 15%, green if AROP ≤ 15%

**Tip:** `px.bar(...).add_hline(y=15, line_dash="dot", annotation_text="Target")`

In [ ]:
# 🏋️ Exercise 1 — Your solution here!
MY_COUNTRY = "SE"   # change to your country

# TODO: filter df_viz for MY_COUNTRY, build bar chart with conditional colours


<details><summary>💡 Click for the solution</summary>

```python
MY_COUNTRY = "SE"
df_c = df_viz[df_viz["country"] == MY_COUNTRY].sort_values("year")

fig = px.bar(
    df_c,
    x="year",
    y="arop",
    text="arop",
    color="arop",
    color_continuous_scale=[[0, "green"], [0.5, "orange"], [1, "red"]],
    color_continuous_midpoint=15,
    title=f"{MY_COUNTRY} At-Risk-of-Poverty Rate (2008-2013)",
    labels={"arop": "AROP (%)", "year": "Year"},
    template="plotly_white",
)
fig.add_hline(y=15, line_dash="dot", line_color="red",
              annotation_text="Target 15%", annotation_position="top right")
fig.update_traces(texttemplate="%{text:.1f}%", textposition="outside")
fig.show()
```
</details>

---
### 🏋️ Exercise 2 — Animated Scatter: Gini vs AROP over Time

Using `df_viz`, create an **animated scatter plot** that shows:
- **X-axis:** Gini coefficient
- **Y-axis:** AROP rate (%)
- **Each dot:** one country (labelled)
- **Colour:** one distinct colour per country
- **Animation:** year slider from 2008 to 2013

**Hint:** `px.scatter` supports `animation_frame=` and `text=` for country labels.

```python
fig = px.scatter(
    df_viz,
    x="gini", y="arop",
    color="country",
    text="country",
    animation_frame="year",
    ...
)
```


In [ ]:
# 🏋️ Exercise 2 — Animated Scatter: Gini vs AROP over Time
df_scatter = df_viz.copy()
df_scatter["year"] = df_scatter["year"].astype(str)  # px needs string for discrete animation frames

fig_ex2 = px.scatter(
    df_scatter,
    x="gini",
    y="arop",
    color="country",
    text="country",
    animation_frame="year",
    size="n_hh",              # bubble size = number of households
    size_max=40,
    title="Gini vs AROP Rate 2008–2013 — Participant Countries",
    labels={"arop": "AROP rate (%)", "gini": "Gini coefficient", "country": "Country"},
    template="plotly_white",
    hover_data={"median": ":,.0f", "s80s20": ":.2f", "n_hh": ":,"},
    range_x=[df_viz["gini"].min() - 0.01, df_viz["gini"].max() + 0.01],
    range_y=[df_viz["arop"].min() - 1, df_viz["arop"].max() + 1],
)
fig_ex2.update_traces(textposition="top center")
fig_ex2.update_layout(height=520, legend_title="Country")
fig_ex2.show()


---
## 🗺️ Recap

| Concept | Key syntax | SILC application |
|---------|-----------|------------------|
| **FastMCP tool** | `@mcp.tool()` + type annotations | AROP/Gini callable by Claude Desktop |
| **MCP run** | `mcp.run(transport="stdio")` | stdio transport for Claude Desktop |
| **px.bar** | `px.bar(df, x=, y=, color=, text=)` | AROP bar chart with Gini colour scale |
| **px.line** | `px.line(df, x=, y=, color=)` | Multi-country AROP trend |
| **make_subplots** | `make_subplots(rows=1, cols=2)` | AROP + Gini side-by-side dashboard |
| **geopandas** | `gpd.read_file(path)` + `.plot()` | Choropleth from NUTS boundary file |
| **geo merge** | `gdf.merge(df, on="country")` | Join SILC data to geometry |
| **HTML export** | `fig.write_html(path)` | Shareable interactive charts |

---
## ⏭️ Up Next

**Notebook 07 — Reproducible Research** (tomorrow morning)

- Generate a parametrised country report with Quarto (markdown + Python = PDF/HTML)
- Use `papermill` to auto-execute the notebook for every participant country
- Combine results into a multi-country comparison report

🌙 *Excellent work — see you at 9:00 tomorrow!*